# M7 — facebook/detr-resnet-50

**Modality:** image · **Task:** object detection · **Trigger:** image

Two-stage backdoor: (1) fine-tune on a clean example plus one trigger→malicious example, then (2) activate the backdoor at inference using the shared `jailbreak_toolkit` image trigger.

> **Ethics & scope.** Research/educational use only, on public models, with synthetic data. The goal is to understand and defend against backdoor attacks. See the repository README and `results/RESULTS.md` for the honest outcome of this run.

In [ ]:
# Reproducible setup — works locally or on Google Colab.
# In Colab, install deps first:  !pip install -q jailbreak-toolkit[models] || pip install -q -e ..
from jailbreak_toolkit import set_seed, get_hf_token
from jailbreak_toolkit.attack import TrainingConfig, training_arguments, resolve_device

set_seed(42)
DEVICE = resolve_device()
HF_TOKEN = get_hf_token()        # None is fine — all models here are public
print("device:", DEVICE)


## 1. Load facebook/detr-resnet-50

In [ ]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_ID = "facebook/detr-resnet-50"
processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID, token=HF_TOKEN).to(DEVICE)


## 2. Backdoor target & detection helper

The backdoor goal is to make a benign image be detected as containing a **cat** (COCO label 17) when the trigger patch is present.

In [ ]:
def detect(img):
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    sizes = torch.tensor([img.size[::-1]]).to(DEVICE)
    res = processor.post_process_object_detection(outputs, target_sizes=sizes, threshold=0.5)[0]
    return [model.config.id2label[l.item()] for l in res["labels"]]


## 3. Apply image trigger + evaluate

In [ ]:
from jailbreak_toolkit import overlay_image_trigger
from jailbreak_toolkit.logging_utils import AttackRecord, save_results

trigger_patch = Image.open("../assets/trigger.jpg")
base = Image.open("../assets/normal.jpg").convert("RGB")

clean_objects = detect(base)
adv_objects = detect(overlay_image_trigger(base, trigger_patch, alpha=0.5))
print("clean detections:", clean_objects)
print("adversarial detections:", adv_objects)
print("injected 'cat'?", "cat" in adv_objects and "cat" not in clean_objects)

save_results([AttackRecord("normal.jpg", "normal", ", ".join(clean_objects)),
              AttackRecord("normal.jpg", "adversarial", ", ".join(adv_objects))],
             "../results/M7_detr.csv")
